Check before running:
1. Check api key is correctly named and stored
2. check the model name
3. Check the TEMPERATURE
4. Check max tokens
5. Check the iterations
6. Paste the description
7. Change the ID of description

In [ ]:
# experiment_runner_mistral_zero_shot

# ── 0. Setup ───────────────────────────────────────────────
!pip install -q --upgrade mistralai pandas

import os

from mistralai.client import Mistral
from google.colab import files
from google.colab import userdata
import pandas as pd
import requests
from lxml import etree
import re

os.makedirs("experiments/dmn", exist_ok=True)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 kB 1.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 996.0/996.0 kB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 76.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.0/220.0 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.4/66.4 kB 4.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.3 which is incompatible.
db-dtypes 1.5.1 requires pandas<3.0.0,>=1.5.3, but you have pandas 3.0.3 which is incompatible.
google-adk 1.29.0 requires opentelemetry-api<1.39.0,>=1.36.0, but you have opentelemetry-api 1.39.1 which is incompatible.
gradio 5.50.0 requires pandas<3.0,>

In [ ]:
# ── 1. API key and model ───────────────────────────────────
API_KEY = userdata.get("MISTRAL_API_KEY")
client = Mistral(api_key=API_KEY)

MODEL_NAME = "mistral-large-2512"

In [ ]:
import requests
from lxml import etree

# ── 2. Configuration ─────────────────
# Change manually per run: 0.2 / 0.4 / 0.6
TEMPERATURE = 0.2

N_ITERATIONS = 3
MAX_TOKENS = 8500
TOP_P = 1.0

# ── DMN Schema and Validation Setup (Moved from cell 0) ─────────────────
# DMN 1.3 Schema URL - Using a common OMG link that works
DMN_SCHEMA_FILE = "DMN13.xsd"
DMN_SCHEMA_URL = "https://www.omg.org/spec/DMN/20191111/DMN13.xsd"

# Download DMN Schema if not exists
xmlschema = None # Initialize xmlschema as None

if not os.path.exists(DMN_SCHEMA_FILE):
    print(f"Attempting to download DMN schema from {DMN_SCHEMA_URL}...")
    try:
        response = requests.get(DMN_SCHEMA_URL)
        response.raise_for_status() # Raise an exception for HTTP errors
        with open(DMN_SCHEMA_FILE, "wb") as f:
            f.write(response.content)
        print("DMN schema downloaded.")

        # Parse the DMN schema once if download successful
        try:
            xmlschema_doc = etree.parse(DMN_SCHEMA_FILE)
            xmlschema = etree.XMLSchema(xmlschema_doc)
            print("DMN schema parsed successfully.")
        except Exception as e:
            print(f"Error parsing DMN schema: {e}. Falling back to basic XML validation.")
            xmlschema = None

    except requests.exceptions.RequestException as e:
        print(f"Error downloading DMN schema from {DMN_SCHEMA_URL}: {e}. Falling back to basic XML validation.")
    except Exception as e:
        print(f"An unexpected error occurred during schema download: {e}. Falling back to basic XML validation.")
else:
    print("DMN schema already exists. Attempting to parse...")
    # Parse the DMN schema once if file exists
    try:
        xmlschema_doc = etree.parse(DMN_SCHEMA_FILE)
        xmlschema = etree.XMLSchema(xmlschema_doc)
        print("DMN schema parsed successfully.")
    except Exception as e:
        print(f"Error parsing DMN schema: {e}. Falling back to basic XML validation.")
        xmlschema = None

def validate_dmn_xml(xml_string: str) -> (bool, list):
    """
    Validates a DMN XML string against the pre-loaded DMN schema, or performs basic
    XML syntax validation if the DMN schema could not be loaded.

    Args:
        xml_string: The DMN XML content as a string.

    Returns:
        A tuple: (is_valid, errors_list). is_valid is True if valid, False otherwise.
        errors_list contains validation errors if any.
    """
    try:
        # First, attempt basic XML parsing to catch malformed XML quickly
        xml_doc = etree.fromstring(xml_string.encode('utf-8'))

        # If DMN schema was loaded, perform full XSD validation
        if xmlschema is not None:
            try:
                xmlschema.assertValid(xml_doc)
                return True, []
            except etree.XMLSchemaError as e:
                return False, [f"DMN Schema Validation Error: {e}"]
        else:
            # If DMN schema could not be loaded, report basic XML validity
            return True, ["Warning: DMN schema not loaded. Performed basic XML syntax check only."]

    except etree.XMLSyntaxError as e:
        return False, [f"XML Syntax Error: {e}"]
    except Exception as e:
        return False, [f"An unexpected error occurred during XML validation: {e}"]

# ── 3. New description ─────────────────
description_id = "description_id"

description = """enter your description here"""
# ── 4. Zero-shot prompt ─────────────────
def build_zero_shot_prompt(description: str) -> str:
    return f"""<s>[INST]

    You are an expert in generating DMN 1.3 XML compatible with Camunda. (Persona)

Generate a complete and executable DMN XML file from the textual description below.(Constraint Instruction)

STRICT OUTPUT RULES:
- Output ONLY XML.
- No explanations, comments, or markdown.
- The XML must start with: <?xml version=\"1.0\" encoding=\"UTF-8\"?>
- The XML must end with </definitions>.
- All tags must be properly closed.

NAMESPACE RULES:
- Use default namespace:
  xmlns=\"https://www.omg.org/spec/DMN/20191111/MODEL/\"
- Include the following namespaces for diagram interchange:
  xmlns:dmndi=\"https://www.omg.org/spec/DMN/20191111/DMNDI/\"
  xmlns:dc=\"http://www.omg.org/spec/DMN/20180521/DC/\"
  xmlns:modeler=\"http://camunda.org/schema/modeler/1.0\"
  xmlns:biodi=\"http://bpmn.io/schema/dmn/biodi/2.0\"
  xmlns:di=\"http://www.omg.org/spec/DMN/20180521/DI/\"

STRUCTURE RULES:
- The root element must be:
  <definitions id=\"Definitions_00ghp5h\" name=\"DRD\" namespace=\"http://example.com/dmn\" exporter=\"Camunda Modeler\" exporterVersion=\"5.44.0\" modeler:executionPlatform=\"Camunda Cloud\" modeler:executionPlatformVersion=\"8.8.0\">

- Include inputData elements when variables are present.
- Each inputData must contain:
  <variable name=\"VARIABLE_NAME\" typeRef=\"VARIABLE_TYPE\"/>

- Include at least one decision:
  <decision id=\"DECISION_ID\" name=\"DECISION_NAME\">

- Each decision must contain exactly one decisionTable.

DECISION TABLE RULES:
- decisionTable must include:
  - one or more <input>
  - exactly one <output name=\"OUTPUT_NAME\" typeRef=\"OUTPUT_TYPE\"/>
  - one or more <rule>
- decisionTable MUST have hitPolicy=\"UNIQUE\":
  <decisionTable id=\"decisionTable_ID\" hitPolicy=\"UNIQUE\">

- Each input must contain:
  <inputExpression id=\"INPUT_EXPRESSION_ID\" typeRef=\"INPUT_TYPE\">
    <text>variable_name</text>
  </inputExpression>

- Each rule must contain:
  - one <inputEntry> per input
  - one <outputEntry>

- inputEntry format:
  <inputEntry><text>FEEL_condition</text></inputEntry>

- Use "-" inside <inputEntry><text>-</text></inputEntry> to represent any value ("don't care").
- Apply this convention consistently in all decision table rules.

- outputEntry format:
  <outputEntry><text>FEEL_value</text></outputEntry>

FEEL RULES:
- Use FEEL unary tests:
  - < 18
  - >= 18
  - [18..65]
- Strings must be in double quotes
- Do not use programming operators like ==, &&, ||

ID RULES:
- All ids must be unique
- Use consistent naming:
  - inputData: input_<name>
  - decision: decision_<name>
  - rules: rule_<number>

QUALITY RULES:
- Ensure the XML is executable in Camunda
- Ensure logic matches the description
- No placeholders or incomplete elements

========================
LAYOUT SYSTEM (CRITICAL)
========================

GRID RULES:
- Row 1 (decision): y = 100
- Row 2 (inputs):   y = 300

INPUT LAYOUT RULES:
- All input nodes on the SAME horizontal row (y = 300)
- x = 100 + (index * 200), index starts at 0
- width = 160, height = 60

DECISION LAYOUT RULES:
- Decision centered above all inputs
- x = midpoint of all input x-coordinates
- y = 100
- width = 180, height = 80

WAYPOINT RULES:
- Edge source = top-center of input shape:
    x = input_x + 80
    y = input_y  ← top edge of input (y=300, NOT y+height)
- Edge target = ALWAYS the same point for ALL edges = bottom-center of decision:
    x = decision_x + 90
    y = decision_y + 80  (bottom edge, since decision height=80)

CRITICAL: Every single DMNEdge must have the IDENTICAL target waypoint.   ← CRITICAL block
All edges share one convergence point at the bottom-center of the decision node.
NEVER use decision_x alone as the target x — always add half the width.
EXAMPLE for decision at x=400, width=180, height=80:
  target waypoint → x=490, y=180

EXAMPLE for input at x=100, width=160:
  source waypoint → x=180, y=300

INPUTDATA LABEL RULES:
- Every <inputData> MUST have a name attribute matching its variable:
  <inputData id=\"input_allergies\" name=\"allergies\">
- NEVER leave name empty

========================
DMNDI RULES:
========================

LOGICAL STRUCTURE (must come BEFORE dmndi section):
- Each <decision> that depends on an inputData must contain:
  <informationRequirement id="IR_<decisionId>_<inputDataId>">
    <requiredInput href="#<inputDataId>"/>
  </informationRequirement>

- Give EVERY informationRequirement a UNIQUE id like "IR_decision1_input_allergies"

DMNDI SECTION:
- Must start with <dmndi:DMNDI>
- Inside: one <dmndi:DMNDiagram id="DMNDiagram_1" name="DRD">

SHAPE RULES:
- For each inputData:
  <dmndi:DMNShape id="DMNShape_<inputDataId>" dmnElementRef="<inputDataId>">
    <dc:Bounds x="X" y="Y" width="125" height="45"/>
  </dmndi:DMNShape>

- For each decision:
  <dmndi:DMNShape id="DMNShape_<decisionId>" dmnElementRef="<decisionId>">
    <dc:Bounds x="X" y="Y" width="180" height="80"/>
  </dmndi:DMNShape>

EDGE RULES (THIS IS THE CRITICAL FIX):
- For each informationRequirement, create ONE DMNEdge:
  <dmndi:DMNEdge id="DMNEdge_<informationRequirementId>"
                 dmnElementRef="<informationRequirementId>">
    <di:waypoint x="SOURCE_CENTER_X" y="SOURCE_BOTTOM_Y"/>
    <di:waypoint x="TARGET_CENTER_X" y="TARGET_TOP_Y"/>
  </dmndi:DMNEdge>

- dmnElementRef on DMNEdge MUST be the informationRequirement id
- NEVER point dmnElementRef to a decision or inputData id on an edge
- Waypoints: source = bottom-center of inputData shape, target = top-center of decision shape


(Format)
Convert the following description into DMN XML: '{description}' [/INST]</s>"""

# ── 5. Clean model output ─────────────────
def clean_model_output(text: str) -> str:
    text = text.strip()
    text = re.sub(r"^```xml\s*", "", text)
    text = re.sub(r"^```\s*", "", text)
    text = re.sub(r"\s*```$", "", text)

    xml_start = text.find("<?xml")
    if xml_start == -1:
        xml_start = text.find("<definitions")

    if xml_start != -1:
        text = text[xml_start:]

    xml_end = text.rfind("</definitions>")
    if xml_end != -1:
        text = text[:xml_end + len("</definitions>")]

    return text.strip()

# ── 5b. Fix DMNEdge references ─────────────────
DMN_NS   = "https://www.omg.org/spec/DMN/20191111/MODEL/"
DMNDI_NS = "https://www.omg.org/spec/DMN/20191111/DMNDI/"
DI_NS    = "http://www.omg.org/spec/DMN/20180521/DI/"

def fix_dmn_edges(xml_string: str) -> str:
    try:
        root = etree.fromstring(xml_string.encode("utf-8"))
    except Exception as e:
        print(f"  [fix] XML parse failed: {e}")
        return xml_string

    # Collect all informationRequirement ids
    ir_map = {}  # ir_id -> (source_ref, decision_id)
    for decision in root.findall(f".//{{{DMN_NS}}}decision"):
        decision_id = decision.get("id")
        for ir in decision.findall(f"{{{DMN_NS}}}informationRequirement"):
            ir_id = ir.get("id")
            req_input = ir.find(f"{{{DMN_NS}}}requiredInput")
            req_decision = ir.find(f"{{{DMN_NS}}}requiredDecision")
            source_ref = None
            if req_input is not None:
                source_ref = req_input.get("href", "").lstrip("#")
            elif req_decision is not None:
                source_ref = req_decision.get("href", "").lstrip("#")
            if ir_id and source_ref:
                ir_map[ir_id] = (source_ref, decision_id)

    # Fix every DMNEdge whose dmnElementRef doesn't point to a valid IR
    used_ir_ids = set()
    fixed = 0
    for edge in root.findall(f".//{{{DMNDI_NS}}}DMNEdge"):
        ref = edge.get("dmnElementRef", "")
        if ref in ir_map:
            used_ir_ids.add(ref)
            continue
        # Assign the next unused IR id
        for ir_id in ir_map:
            if ir_id not in used_ir_ids:
                edge.set("dmnElementRef", ir_id)
                used_ir_ids.add(ir_id)
                fixed += 1
                break

    if fixed:
        print(f"  [fix] Patched {fixed} DMNEdge dmnElementRef(s)")

    return etree.tostring(root, pretty_print=True,
                          xml_declaration=True, encoding="UTF-8").decode("utf-8")

# ── 6. Run generation and validation ─────────────────
prompt = build_zero_shot_prompt(description)

valid_dmn_files_for_download = []

for iteration in range(1, N_ITERATIONS + 1):
    print(f"▶ Mistral zero-shot | {description_id} | temp={TEMPERATURE} | iter={iteration}")

    response = client.chat.complete(
        model=MODEL_NAME,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=TEMPERATURE,
        max_tokens=MAX_TOKENS,
        top_p=TOP_P,
    )

    usage = response.usage

    print("Input tokens:", getattr(usage, "prompt_tokens", "N/A"))
    print("Output tokens:", getattr(usage, "completion_tokens", "N/A"))
    print("Total tokens:", getattr(usage, "total_tokens", "N/A"))

    dmn_xml = clean_model_output(response.choices[0].message.content)
    dmn_xml = fix_dmn_edges(dmn_xml)

    base_name = f"{description_id}_mistral_zero_shot_temp_{TEMPERATURE}_iter_{iteration}"
    dmn_path = f"experiments/dmn/{base_name}.dmn"

    with open(dmn_path, "w", encoding="utf-8") as f:
        f.write(dmn_xml)

    print(f"Saved: {dmn_path}")

    is_valid, errors = validate_dmn_xml(dmn_xml)

    if is_valid:
        print("✅ DMN XML is valid.")
        valid_dmn_files_for_download.append(dmn_path)
    else:
        print("❌ DMN XML is INVALID. Errors:")
        for error in errors:
            print(f"  - {error}")

# ── 7. Download DMN results (only valid ones) ─────────────────
for dmn_file_path in valid_dmn_files_for_download:
    files.download(dmn_file_path)


DMN schema already exists. Attempting to parse...
Error parsing DMN schema: Element '{http://www.w3.org/2001/XMLSchema}element', attribute 'ref': The QName value '{https://www.omg.org/spec/DMN/20191111/DMNDI/}DMNDI' does not resolve to a(n) element declaration., line 55. Falling back to basic XML validation.
▶ Mistral zero-shot | description_3 | temp=0.4 | iter=1
Input tokens: 1828
Output tokens: 4479
Total tokens: 6307
  [fix] XML parse failed: StartTag: invalid element name, line 271, column 18 (<string>, line 271)
Saved: experiments/dmn/description_3_mistral_zero_shot_temp_0.4_iter_1.dmn
❌ DMN XML is INVALID. Errors:
  - XML Syntax Error: StartTag: invalid element name, line 271, column 18 (<string>, line 271)
▶ Mistral zero-shot | description_3 | temp=0.4 | iter=2
Input tokens: 1828
Output tokens: 5543
Total tokens: 7371
  [fix] XML parse failed: StartTag: invalid element name, line 441, column 18 (<string>, line 441)
Saved: experiments/dmn/description_3_mistral_zero_shot_temp_0.4_